##### next steps

* validate production of sentinel_request_df. are all values coming through as intended?

* dial in use of resx and resy parameters (see [link](https://docs.sentinel-hub.com/api/latest/reference/#tag/process/operation/process))
    * may be as simple as requesting in proper utm epsg and specifying resx and resy to equal sentinel 2's native meters by meters?
    * this page might be helpful: [link](https://www.sentinel-hub.com/faq/how-can-i-get-higher-resolution/)
    * monitor how this impacts pu's spent
* build out eval scripts and eval script organization strategy
    * start with atmos penetration, then get extra nerdy
    * how would cloud cover impact logic for using different eval scripts? would it?
    * monitor how different evals impact pu's spent
* build out file naming strategy. must encompass:
    * image timestamp
    * fire identifier (ak fire number? irwin?)
    * image identifier (sent2_l2a, for example)
    * evalscript identifier (true color, atmospheric penetration, etc.)
* build out warnings & error tracking
    * edge case where multiple tiles with disparate time stamps are used for the orbit mosaic? doesnt seem possible with sentinel 2 given the orbit cycle.

In [58]:
import asyncio
from collections import defaultdict
import geopandas as gpd
import json
from pathlib import Path
import pandas as pd
from pyproj import CRS, Transformer
import shapely as shp
import shutil

from utils.arcgis_helpers import AsyncArcGISRequester, arcgis_features_to_gdf, checkout_token
from utils.sentinel_helpers import EVALSCRIPT, AsyncSentinelRequester, checkout_sentinel_token, parse_response_list

In [2]:
nifc_token = checkout_token('NIFC_AGO', 60, 'NIFC_TOKEN', 15)

sentinel_token = Path.cwd() / 'secrets' / 'sentinel_token.txt'
sentinel_token = checkout_sentinel_token(sentinel_token, 'PLANET_OAUTH_ID', 'PLANET_OAUTH_SECRET', 30)

In [ ]:
async with AsyncArcGISRequester() as requester:

    # if 1 mile buffer bbox is too large as a default, query perims&locs layer and create buffer bbox locally
    # that 1 mile zone may be nice, even for little fires. just see what the earth is looking like. do some fancy eval scripts.
    ak_current_fires = await requester.arcgis_rest_api_get(
        base_url=r'https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/TESTING_AK_Wildfire_Values_at_Risk/FeatureServer/1',
        params={
            'f': 'json',
            'token': nifc_token,
            'outfields': 'AkFireNumber,wfigs_IncidentName,wfigs_IrwinID,Sentinel2_L2A_LastImage_dt',
            'where': "FireActivityStatus <> 'Out'",
            'returnEnvelope': 'true',
            'returnGeometry': 'false'
        },
        operation='query?'
    )

    current_irwins = [feat['attributes']['wfigs_IrwinID'] for feat in ak_current_fires['features']]
    wfigs_discovery_dt = await requester.arcgis_rest_api_get(
        base_url=r'https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Incident_Locations_YearToDate/FeatureServer/0',
        params={
            'f': 'json',
            'outfields': 'IrwinID,FireDiscoveryDateTime',
            'where': f"IrwinID IN ({','.join(f"'{irwin}'" for irwin in current_irwins)})",
            'returnGeometry': 'false'
        },
        operation='query?'
    )

    xmins = [feat['envelope']['xmin'] for feat in ak_current_fires['features']]
    ymins = [feat['envelope']['ymin'] for feat in ak_current_fires['features']]
    xmaxs = [feat['envelope']['xmax'] for feat in ak_current_fires['features']]
    ymaxs = [feat['envelope']['xmax'] for feat in ak_current_fires['features']]
    utm_zones = await requester.arcgis_rest_api_get(
        base_url=r'https://services.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/World_UTM_Grid/FeatureServer/0',
        params={
            'f': 'json',
            'outfields': 'ZONE',
            'geometry': json.dumps({'xmin':min(xmins), 'ymin':min(ymins), 'xmax':max(xmaxs), 'ymax':max(ymaxs)}),
            'geometryType': 'esriGeometryEnvelope',
            'inSR': 3338,
            'outSR': 3338,
            'spatialRel': 'esriSpatialRelIntersects',
            'returnGeometry': 'true'
        },
        operation='query?'
    )

Entering async context...
Exiting async context...


In [ ]:
#! this is a cleaner approach to existing functions in the arcgis_helpers module
def json_features_to_dataframe(features: dict) -> pd.DataFrame:
    df_dict = defaultdict(list)
    for feat in features['features']:
        for attrName, attrVal in feat['attributes'].items():
            df_dict[attrName].append(attrVal)
        for key, val in feat.items():
            if key == 'attributes':
                continue
            df_dict[key].append(val)
    return pd.DataFrame(df_dict)

In [99]:
# create gdf of ak current fires with aoi bounding boxes for geometries
ak_current_fires_df = json_features_to_dataframe(ak_current_fires)
ak_current_fires_df['bbox'] = ak_current_fires_df['envelope'].apply(
    lambda x: [round(coord, 3) for coord in (x['xmin'], x['ymin'], x['xmax'], x['ymax'])]
)
ak_current_fires_df['bbox_geometry'] = ak_current_fires_df['bbox'].apply(
    lambda x: shp.box(*x)
)
ak_current_fires_gdf = gpd.GeoDataFrame(ak_current_fires_df, geometry='bbox_geometry', crs='EPSG:3338')

# determine which utm zone maximally intersects each fires aoi bounding box
utm_zones_gdf = arcgis_features_to_gdf(utm_zones)
ak_current_fires_gdf = ak_current_fires_gdf.sjoin(utm_zones_gdf)
ak_current_fires_gdf['utm_zone_geometry'] = utm_zones_gdf.loc[ak_current_fires_gdf['index_right'], 'geometry'].values
ak_current_fires_gdf['bbox_utm_zone_intersection'] = ak_current_fires_gdf.apply(
    lambda x: shp.intersection(x['bbox_geometry'], x['utm_zone_geometry']).area,
    axis=1
)
ak_current_fires_gdf.sort_values('bbox_utm_zone_intersection', inplace=True, ascending=False)
ak_current_fires_gdf.drop_duplicates(['AkFireNumber','wfigs_IncidentName'], inplace=True)

# create a bounding box in the proper utm zone projection for each fire
ak_current_fires_gdf['utm_zone'] = ak_current_fires_gdf['ZONE'].apply(
    lambda x: f'326{str(x).zfill(2)}'
)
crs_3338 = CRS('EPSG:3338')
transformer_dict = {utm_zone: Transformer.from_crs(crs_3338, CRS(f'EPSG:{utm_zone}')) for utm_zone in ak_current_fires_gdf['utm_zone']}
ak_current_fires_gdf['utm_zone_bbox'] = ak_current_fires_gdf.apply(
    lambda x: transformer_dict[x['utm_zone']].transform_bounds(*x['bbox']),
    axis=1
)
ak_current_fires_gdf['utm_zone_bbox'] = ak_current_fires_gdf['utm_zone_bbox'].apply(
    lambda x: [round(coord, 3) for coord in x]
)

# formatting the sentinel request dataframe
wfigs_discovery_dt_df = json_features_to_dataframe(wfigs_discovery_dt)
sentinel_request_df = pd.merge(ak_current_fires_gdf, wfigs_discovery_dt_df, left_on='wfigs_IrwinID', right_on='IrwinID')
sentinel_request_df['image_dt_from'] = sentinel_request_df.apply(
    lambda x: x['Sentinel2_L2A_LastImage_dt'] if x['Sentinel2_L2A_LastImage_dt'] else x['FireDiscoveryDateTime'],
    axis=1
)
sentinel_request_df['fire'] = sentinel_request_df.apply(
    lambda x: f'{str(x['AkFireNumber']).zfill(3)}-{x['wfigs_IncidentName'].replace(' ','-')}',
    axis=1
)
sentinel_request_df = sentinel_request_df[['fire','utm_zone_bbox','utm_zone','image_dt_from']]

In [100]:
sentinel_request_df

,fire,utm_zone_bbox,utm_zone,image_dt_from
0,020-Sacaloff,"[609691.722, 6702631.009, 612989.204, 6705902....",32605,1743960423000
1,038-Lazy-Mountain,"[388305.846, 6830885.698, 391878.922, 6834434....",32606,1745642880000
2,040-Bald-Eagle,"[370293.967, 6832133.956, 373867.41, 6835683.594]",32606,1745682628000


In [ ]:
async with AsyncSentinelRequester() as requester:
    fire_orbits = await asyncio.gather(
        *(requester.sentinel_l2a_orbits(
            bbox=bbox,
            bbox_epsg=epsg,
            epoch_milliseconds_from=image_dt_from,
            evalscript=EVALSCRIPT['true_color'],
            token=sentinel_token,
            aoi_identifier=fire
        ) for fire, bbox, epsg, image_dt_from in sentinel_request_df.itertuples(index=False))
    )
    

In [ ]:
sample_objs_dir = Path.cwd() / 'sample_objects'

for orbit in fire_orbits:
    response, pu_spent, identifier = orbit
    responses_with_images = parse_response_list(response)

    image_timestamps = []
    for tup in responses_with_images:
        dates, tif = tup
        
        # this should log a warning
        assert len(dates) == 1, 'multiple tile dates found for single returned orbit image!'

        tif_name = f'{identifier}_{dates[0].replace(':','-')}_sent2-l2a_true-color'

        with open(sample_objs_dir / f'{tif_name}.tiff', 'wb') as file:
            shutil.copyfileobj(tif, file)

        # convert to epoch milliseconds utc
        # maximum value will be used in applyEdits request to update attribute in ak wf var service
        image_timestamps.append(dates[0])
    